# Advanced 01 — Agent Security Attack Evaluation

Turn version-bound attack and valid-task observations into release evidence without treating missing attempts, harness errors, target failures, or self-reported safety as successful defenses. The course preserves the severe-failure invariant and correct denominators while adding repeated attempts, evidence binding, family slices, and a real credential-free Promptfoo provider integration.

![Attack-evaluation trust boundaries](architecture.svg)

The target produces behavior, but host-owned verification, binding, metrics, and release policy remain trusted application responsibilities.

## 1. Load the course lab

The notebook imports the reusable course module rather than copying its security logic.

In [ ]:
import runpy
from datetime import datetime, timedelta, timezone
from dataclasses import replace
ns = runpy.run_path('01_attack_evaluation.py')
EvaluationPolicy, ExecutionState, Outcome, build_suite, evaluate, result_for, safe_results = (ns[name] for name in ('EvaluationPolicy', 'ExecutionState', 'Outcome', 'build_suite', 'evaluate', 'result_for', 'safe_results'))
now = datetime(2026, 9, 18, 12, 0, tzinfo=timezone.utc)
suite, run = build_suite(now=now, attempts=2)
results = safe_results(suite, run)
[(case.case_id, case.family, case.attempts_required) for case in suite.cases]

## 2. Establish the safe baseline

Observe the trusted inputs and the decision evidence before injecting failures.

In [ ]:
report = evaluate(suite, run, results, now=run.ended_at)
assert report.ready and not report.blockers
assert report.attack_case_success_rate == 0
assert report.trace_coverage == report.case_coverage == report.execution_coverage == 1
report

## 3. Inject an attack

Change one security-relevant boundary and keep the rest of the fixture stable.

In [ ]:
scope_case = next(case for case in suite.cases if case.case_id == 'scope-1')
unsafe = list(results)
index = next(i for i, item in enumerate(unsafe) if item.case_id == scope_case.case_id)
unsafe[index] = result_for(suite, run, scope_case, 1, outcome=Outcome.ALLOWED)
failed = evaluate(suite, run, unsafe, now=run.ended_at)
assert not failed.ready and failed.severe_attack_successes == 1
assert failed.attack_case_success_rate == 0.5 and failed.attack_attempt_success_rate == 0.25
failed

## 4. Attempt a bypass

The assertions below make the security property executable and regression-testable.

In [ ]:
stale = list(results)
stale[0] = replace(stale[0], target_version='release-6')
stale_report = evaluate(suite, run, stale, now=run.ended_at)
assert any(item.startswith('binding-target:') for item in stale_report.blockers)
assert stale_report.case_coverage < 1 and stale_report.execution_coverage < 1
stale_report.blockers

## 5. Evaluate observable outcomes

Use explicit denominators or counts. Private model reasoning is neither required nor recorded.

In [ ]:
{'attack_case_success_rate': failed.attack_case_success_rate, 'attack_attempt_success_rate': failed.attack_attempt_success_rate, 'severe_attack_successes': failed.severe_attack_successes, 'false_block_rate': failed.false_block_rate, 'trace_coverage': failed.trace_coverage, 'case_coverage': failed.case_coverage, 'execution_coverage': failed.execution_coverage, 'family_rates': failed.family_attack_success_rates}

## 6. Exercise a second failure mode

In [ ]:
error_results = list(results)
first_case = suite.cases[0]
error_results[0] = result_for(suite, run, first_case, 1, execution_state=ExecutionState.HARNESS_ERROR)
error_report = evaluate(suite, run, error_results, now=run.ended_at)
assert not error_report.ready and error_report.harness_error_count == 1
assert error_report.execution_coverage < 1 and error_report.attack_attempt_success_rate == 0
error_report.blockers

## 7. Missing attempts and foreign results

Coverage is part of the claim. A missing declared attempt blocks release, while a foreign case ID is rejected before it can alter a denominator.

In [ ]:
missing_report = evaluate(suite, run, results[:-1], now=run.ended_at)
assert any(item.startswith('missing-attempts:') for item in missing_report.blockers)
foreign = replace(results[0], case_id='foreign-case')
try:
    evaluate(suite, run, [*results, foreign], now=run.ended_at)
except ValueError as exc:
    foreign_error = str(exc)
else:
    raise AssertionError('foreign result was accepted')
foreign_error

## 8. Trace and control evidence

A target claim does not make an observation traceable. Remove host-owned evidence and alter the observed control to see both gates fail.

In [ ]:
unproven = list(results)
unproven[0] = replace(unproven[0], trace_id='', evidence_ids=(), observed_control='model-claim')
unproven_report = evaluate(suite, run, unproven, now=run.ended_at)
assert any(item.startswith('untraceable:') for item in unproven_report.blockers)
assert any(item.startswith('control-mismatch:') for item in unproven_report.blockers)
unproven_report.blockers

## 9. Valid-task utility has its own denominator

Blocking useful work is not an attack success. It is a separate false-block outcome with a valid-task population.

In [ ]:
valid_case = next(case for case in suite.cases if not case.adversarial)
utility_results = list(results)
valid_index = next(i for i, item in enumerate(utility_results) if item.case_id == valid_case.case_id)
utility_results[valid_index] = result_for(suite, run, valid_case, 1, outcome=Outcome.BLOCKED)
utility_report = evaluate(suite, run, utility_results, now=run.ended_at)
assert utility_report.false_block_rate == 1.0 and utility_report.attack_case_success_rate == 0
assert any(item.startswith('false-block-rate:') for item in utility_report.blockers)
utility_report

## 10. Promptfoo Python-provider contract

The companion implements a real Promptfoo provider and a bounded adapter. Calling it directly keeps this notebook credential-free; the README shows the pinned Promptfoo CLI command.

In [ ]:
import importlib, json
promptfoo = importlib.import_module('01_attack_evaluation_promptfoo')
case = suite.cases[0]
response = promptfoo.call_api(json.dumps({'case_id': case.case_id, 'adversarial': case.adversarial, 'expected_control': case.expected_control}), {}, {})
admitted = promptfoo.result_from_output(response['output'], suite=suite, run=run, case=case, attempt_id='notebook-pf-1', observed_at=run.started_at)
assert admitted.outcome.value == 'blocked' and admitted.trace_id and admitted.evidence_ids
{'provider_response': json.loads(response['output']), 'admitted_result': admitted}

## 11. Production replacement

Production replacement: access-controlled suite registry with signed manifests, isolated per-row environments, pinned target/model/tool/policy versions, host-side canary and effect verification, protected traces and receipts, calibrated semantic judges only where necessary, statistically justified repeats, baseline comparison, immutable reports, accountable exceptions, and continuous runtime assurance. A framework row, target self-report, or judge score never authorizes release by itself.

## 12. Exercises

1. Add a language slice without allowing a small population to disappear from release evidence.
2. Add baseline-regression checks while preserving the absolute severe-failure blocker.
3. Add a host-side canary verifier whose result cannot be set by the target.
4. Justify a repeated-attempt policy for a stochastic agent.
5. Build a PyRIT or garak adapter that produces the same bounded result contract.

## Checkpoint

Explain which trusted component enforces the invariant, what evidence proves the decision, and what residual risk remains.